# Layer 3: reliability with nested selection

In [ ]:
from __future__ import annotations

import sys
import json
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

for _cand in (Path.cwd() / "notebooks", Path.cwd(), Path.cwd().parent):
    if (_cand / "layer3_reliability.py").exists():
        sys.path.insert(0, str(_cand))
        break
import layer3_reliability as rel
import crossfarm_nested as cn

ROOT = cn.ROOT
NESTED = ROOT / "output" / "14n_layer2_crossfarm_nested"
CNN_DIR = ROOT / "output" / "cnn_baseline"
OUT_CALIB = ROOT / "output" / "19n_layer3_reliability_analysis"
OUT_SELECT = ROOT / "output" / "20n_layer3_selective_prediction"
OUT_DICT = ROOT / "output" / "21n_layer3_dictionary"
OUT_SUMMARY = ROOT / "output" / "22n_layer3_confidence_review"
for d in (OUT_CALIB, OUT_SELECT, OUT_DICT, OUT_SUMMARY):
    d.mkdir(parents=True, exist_ok=True)

TARGETS = ["is_nonvocal", "is_distress_related", "is_resource_related", "proxy_functional_v1"]
COVERAGES = [1.0, 0.95, 0.90, 0.85, 0.80, 0.75, 0.70, 0.60, 0.50]
RUN_TS = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

In [ ]:
clips = cn.load_clip_table()
fw_outer = pd.read_csv(NESTED / "crossfarm_fold_predictions.csv.gz")
fold_settings = {int(k): v for k, v in json.load(open(NESTED / "crossfarm_manifest.json"))["fold_settings"].items()}
fw_inner, inner_check = cn.inner_predictions(clips, fold_settings, fw_outer)

sel = pd.read_csv(NESTED / "crossfarm_bundle_selection_scores.csv", dtype={"heldout_farm": str})
inner_check = inner_check.merge(sel, on=["heldout_farm", "target_name", "bundle_name"])
inner_check["abs_diff"] = (inner_check["inner_macro_f1"] - inner_check["inner_macro_f1_recomputed"]).abs()
assert inner_check["abs_diff"].max() < 1e-9, inner_check
fw_inner.to_csv(OUT_CALIB / "framework_inner_predictions.csv.gz", index=False, compression="gzip")
print("inner predictions:", len(fw_inner), "rows; selection scores reproduced (max diff",
      inner_check["abs_diff"].max(), ")")

In [ ]:
fw_o = rel.standardise_predictions(fw_outer, "target_name", "heldout_farm")
fw_i = rel.standardise_predictions(fw_inner, "target_name", "heldout_farm", "inner_val_farm")
(fw_store, calibration_df, selective_df, coverage_df,
 fw_folds, fw_pooled) = rel.summarise_model(fw_o, fw_i, TARGETS, COVERAGES)

cnn_outer = pd.read_csv(CNN_DIR / "cnn_lofo_predictions.csv.gz")
cnn_inner = pd.read_csv(CNN_DIR / "cnn_lofo_inner_predictions.csv.gz")
cnn_o = rel.standardise_predictions(cnn_outer, "target", "held_farm")
cnn_i = rel.standardise_predictions(cnn_inner, "target", "held_farm", "inner_val_farm")
(cnn_store, cnn_calib, cnn_select, cnn_curves,
 cnn_folds, cnn_pooled) = rel.summarise_model(cnn_o, cnn_i, TARGETS, COVERAGES)

In [ ]:
def compare(calib, select, pooled, model):
    c = calib.set_index("target"); s = select.set_index("target"); p = pooled.set_index("target")
    rows = []
    for t in TARGETS:
        rows.append({"model": model, "target": t,
                     "ece_raw": c.loc[t, "ece_raw"],
                     "ece_calibrated_nested": c.loc[t, "ece_selected"],
                     "ece_calibrated_pooled_selection": p.loc[t, "ece_selected"],
                     "method_nested": c.loc[t, "selected_method"], "method_pooled_selection": p.loc[t, "selected_method"],
                     "score_nested": s.loc[t, "score_version"], "score_pooled_selection": p.loc[t, "score_version"],
                     "retained_macro_f1_80cov_nested": s.loc[t, "retained_macro_f1_80cov"],
                     "retained_macro_f1_80cov_pooled_selection": p.loc[t, "retained_macro_f1_80cov"],
                     "baseline_macro_f1_nested": s.loc[t, "baseline_macro_f1"]})
    return pd.DataFrame(rows)

nested_vs_pooled = pd.concat([compare(calibration_df, selective_df, fw_pooled, "framework"),
                              compare(cnn_calib, cnn_select, cnn_pooled, "cnn")], ignore_index=True)

fw_l2 = fw_o.groupby("target").apply(lambda d: f1_score(d["y"], rel.prediction_matrix(d)[0].argmax(1), average="macro", zero_division=0))
cnn_l2 = cnn_o.groupby("target").apply(lambda d: f1_score(d["y"], rel.prediction_matrix(d)[0].argmax(1), average="macro", zero_division=0))
cnn_reliability = pd.DataFrame([{
    "target": t, "cnn_lofo_macro_f1": cnn_l2[t],
    "cnn_ece_raw": cnn_calib.set_index("target").loc[t, "ece_raw"],
    "cnn_ece_calibrated": cnn_calib.set_index("target").loc[t, "ece_selected"],
    "cnn_calibration_method": cnn_calib.set_index("target").loc[t, "selected_method"],
    "cnn_retained_macro_f1_80cov": cnn_select.set_index("target").loc[t, "retained_macro_f1_80cov"],
    "cnn_abstention_rate": cnn_select.set_index("target").loc[t, "abstention_rate"],
    "cnn_selective_score": cnn_select.set_index("target").loc[t, "score_version"]} for t in TARGETS])

framework_vs_cnn = pd.DataFrame([{
    "target": t,
    "framework_lofo_macro_f1": fw_l2[t], "cnn_lofo_macro_f1": cnn_l2[t],
    "framework_ece_raw": calibration_df.set_index("target").loc[t, "ece_raw"],
    "framework_ece_calibrated": calibration_df.set_index("target").loc[t, "ece_selected"],
    "cnn_ece_raw": cnn_calib.set_index("target").loc[t, "ece_raw"],
    "cnn_ece_calibrated": cnn_calib.set_index("target").loc[t, "ece_selected"],
    "framework_retained_macro_f1_80cov": selective_df.set_index("target").loc[t, "retained_macro_f1_80cov"],
    "cnn_retained_macro_f1_80cov": cnn_select.set_index("target").loc[t, "retained_macro_f1_80cov"]}
    for t in TARGETS])

In [ ]:
token_counts = pd.read_csv(ROOT / "output" / "05_tokens_refined" / "token_counts_by_clip.csv")
unit_membership = token_counts[["clip_id", "unit_id"]].drop_duplicates()

dictionary_rows, tier_distribution = [], []
for target in TARGETS:
    r = fw_store[target]
    conf = r["sel_P"].max(1)
    clip_conf = pd.DataFrame({"clip_id": r["clip_ids"], "confidence": conf})
    clip_conf["uncertain"] = rel.deferred_mask(conf, 0.80).astype(int)
    merged = unit_membership.merge(clip_conf, on="clip_id", how="inner")
    counts = {"HIGH": 0, "MEDIUM": 0, "LOW": 0, "UNCERTAIN": 0}
    for unit_id, grp in merged.groupby("unit_id"):
        unc_rate = grp["uncertain"].mean()
        tier = ("HIGH" if unc_rate <= 0.15 else "MEDIUM" if unc_rate <= 0.35
                else "LOW" if unc_rate <= 0.50 else "UNCERTAIN")
        counts[tier] += 1
        dictionary_rows.append({"target": target, "unit_id": unit_id, "n_clips": len(grp),
                                "mean_confidence": grp["confidence"].mean(), "uncertain_rate": unc_rate,
                                "tier": tier})
    total = sum(counts.values())
    tier_distribution.append({"target": target, "n_units": total,
                              **{f"{k}_pct": round(100 * v / total, 1) for k, v in counts.items()}})
dictionary_df = pd.DataFrame(dictionary_rows)
tier_distribution_df = pd.DataFrame(tier_distribution)

In [ ]:
master_idx = pd.read_csv(ROOT / "output" / "09_layer2_dataset" / "layer2_master_table.csv.gz").set_index("_record_key")


def _norm_entropy(P):
    Pc = np.clip(P, 1e-12, 1.0)
    return (-np.sum(Pc * np.log(Pc), axis=1)) / np.log(P.shape[1])


confidence_rows, perfarm_rows, reason_rows, polysemous_rows = [], [], [], []
for target in TARGETS:
    r = fw_store[target]
    y, farms, clip_ids = r["y"], r["farms"], r["clip_ids"]
    scoreP, raw_P, cal_P = r["sel_P"], r["raw_P"], r["cal_P"]
    yp = scoreP.argmax(1)
    raw_pred = raw_P.argmax(1)
    correct = (yp == y).astype(int)
    sel_conf = scoreP.max(1)
    raw_conf, raw_ok = raw_P.max(1), (raw_pred == y)
    confidence_rows.append({"target": target,
                            "mean_raw_confidence": float(raw_conf.mean()),
                            "mean_calibrated_confidence": float(cal_P.max(1).mean()),
                            "raw_confidence_correct": float(raw_conf[raw_ok].mean()),
                            "raw_confidence_incorrect": float(raw_conf[~raw_ok].mean()),
                            "raw_accuracy": float(raw_ok.mean()),
                            "selected_score_confidence_correct": float(sel_conf[correct == 1].mean()),
                            "selected_score_confidence_incorrect": float(sel_conf[correct == 0].mean()),
                            "selected_score_accuracy": float(correct.mean()),
                            "mean_normalised_entropy_selected_score": float(_norm_entropy(scoreP).mean())})
    for f in sorted(set(farms.tolist())):
        m = farms == f
        perfarm_rows.append({"target": target, "heldout_farm": f, "n_clips": int(m.sum()),
                             "raw_accuracy": float((raw_pred[m] == y[m]).mean()),
                             "selected_score_accuracy": float((yp[m] == y[m]).mean())})
    ent = _norm_entropy(scoreP)
    ms = master_idx.reindex(clip_ids)
    dominance = (ms["top_token_count"] / ms["n_segments"]).to_numpy()
    diversity = ms["n_unique_tokens"].to_numpy()
    flags = {"high_predictive_entropy": ent > np.quantile(ent, 0.75),
             "low_confidence": rel.deferred_mask(sel_conf, 0.80),
             "repeated_token_dominance": dominance > 0.5,
             "low_token_diversity": diversity <= 3}
    for reason, mask in flags.items():
        reason_rows.append({"target": target, "reason": reason,
                            "n_clips": int(np.nansum(np.asarray(mask, dtype=float)))})
    clip_df = pd.DataFrame({"clip_id": clip_ids, "pred": yp, "entropy": ent})
    mu = unit_membership.merge(clip_df, on="clip_id", how="inner")
    stats = mu.groupby("unit_id").agg(mean_entropy=("entropy", "mean"),
                                      dominant_share=("pred", lambda s: s.value_counts(normalize=True).max()),
                                      n_clips=("pred", "size")).reset_index()
    ent_cut = stats["mean_entropy"].quantile(0.85)
    poly = stats[(stats["mean_entropy"] >= ent_cut) & (stats["dominant_share"] < 0.60)]
    polysemous_rows.append({"target": target, "n_polysemous": int(len(poly)),
                            "polysemous_units": ";".join(poly["unit_id"].astype(str).tolist())})

confidence_summary_df = pd.DataFrame(confidence_rows)
perfarm_accuracy_df = pd.DataFrame(perfarm_rows)
uncertainty_reasons_df = pd.DataFrame(reason_rows)
polysemous_df = pd.DataFrame(polysemous_rows)

In [ ]:
master_df = pd.DataFrame([{
    "target": t, "lofo_macro_f1": round(fw_l2[t], 3),
    "calibration_method": calibration_df.set_index("target").loc[t, "selected_method"],
    "ece_raw": round(calibration_df.set_index("target").loc[t, "ece_raw"], 3),
    "ece_selected": round(calibration_df.set_index("target").loc[t, "ece_selected"], 3),
    "selective_score": selective_df.set_index("target").loc[t, "score_version"],
    "retained_macro_f1_80cov": round(selective_df.set_index("target").loc[t, "retained_macro_f1_80cov"], 3),
    "selective_gain": round(selective_df.set_index("target").loc[t, "selective_gain"], 3)} for t in TARGETS])

calibration_df.to_csv(OUT_CALIB / "calibration_summary.csv", index=False)
fw_folds.to_csv(OUT_CALIB / "calibration_selection_by_fold.csv", index=False)
cnn_folds.to_csv(OUT_CALIB / "cnn_calibration_selection_by_fold.csv", index=False)
nested_vs_pooled.to_csv(OUT_CALIB / "nested_vs_pooled_selection.csv", index=False)
selective_df.to_csv(OUT_SELECT / "selective_prediction_summary.csv", index=False)
coverage_df.to_csv(OUT_SELECT / "selective_coverage_curves.csv", index=False)
cnn_curves.to_csv(OUT_SELECT / "cnn_selective_coverage_curves.csv", index=False)
cnn_reliability.to_csv(OUT_SELECT / "cnn_reliability_summary.csv", index=False)
framework_vs_cnn.to_csv(OUT_SELECT / "framework_vs_cnn_reliability.csv", index=False)
dictionary_df.to_csv(OUT_DICT / "dictionary_unit_tiers.csv", index=False)
tier_distribution_df.to_csv(OUT_DICT / "dictionary_tier_distribution.csv", index=False)
polysemous_df.to_csv(OUT_DICT / "polysemous_candidates.csv", index=False)
master_df.to_csv(OUT_SUMMARY / "layer3_reliability_master_summary.csv", index=False)
confidence_summary_df.to_csv(OUT_SUMMARY / "confidence_summary.csv", index=False)
perfarm_accuracy_df.to_csv(OUT_SUMMARY / "per_farm_accuracy.csv", index=False)
uncertainty_reasons_df.to_csv(OUT_SUMMARY / "uncertainty_reasons_summary.csv", index=False)

with open(OUT_SUMMARY / "layer3_manifest.json", "w", encoding="utf-8") as f:
    json.dump({"build_ts_utc": RUN_TS, "targets": TARGETS, "coverages": COVERAGES,
               "prediction_source": "14n_layer2_crossfarm_nested/crossfarm_fold_predictions.csv.gz",
               "selection": "calibrator and score chosen per held-out farm from out-of-farm predictions "
                            "inside its training partition"}, f, indent=2)

In [ ]:
pd.set_option("display.width", 250)
display(fw_folds.round(3))
display(cnn_folds.round(3))
display(master_df)
display(nested_vs_pooled.round(3))
display(framework_vs_cnn.round(3))
display(tier_distribution_df)